<a href="https://colab.research.google.com/github/LiamCarter2000/Clinical-Data-Related-Projects/blob/main/3%2B3_Simulations_orig_2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [2]:
import pandas as pd
import numpy as np
import math
import heapq
from typing_extensions import final
from tabulate import tabulate
from collections import defaultdict
from dataclasses import dataclass, replace
from typing import List, Optional

#Data Processing

In [3]:
df = pd.read_csv("Decision Grid.csv", usecols=[1, 2, 3, 4, 5, 6, 7, 8])

df.columns = ['Treated', 'DLT', 'Pending', 'State', 'Orig_33_Action', 'IQ_33_Action', 'Orig_6_Action', 'IQ_6_Action']

decision_dict_IQ_33 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['IQ_33_Action'].to_dict()
decision_dict_33 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['Orig_33_Action'].to_dict()
decision_dict_IQ_6 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['IQ_6_Action'].to_dict()
decision_dict_6 = df.set_index(['Treated', 'DLT', 'Pending', 'State'])['Orig_6_Action'].to_dict()

#Dataclass


Data is processed into four python dictionaries: one for each of the methodologies for which the simulation will run (original 3+3, IQ 3+3, original rolling 6, IQ rolling 6). The values in each of the columns of these methodologies are as follows.

 Defines what dose level the next patient should be treated at, based on the current number of patients being treated, the number of reported DLTs, and the number of results pending at the current dose level.

Possible Dispositions:

1. Up          = start testing patients at the next higher dose level
2. Down     = start testing patients at the next lower dose level
3. Stay       = keep testing patients at the current dose level
4. Suspend = wait for additional testing to finish at current dose
                      level, before starting a test on another patient.
5. MTD       = current dose is the Maximum Tolerated Dose (MTD)


The value of having this as a python dictionary is that all values can be looked up in place for a much faster run time. So if we want to know what to do for IQ 3+3 when we have Treated = 3, DLT = 0, Pending = 1, and State = open, we can just query the corresponding dictionary with "decision_dict_IQ_33.get((3, 0, 1, 'open'))" to get "Stay"


In [4]:
@dataclass
class TrialConfig:
  starting_dose: int = 2
  min_dose: int = 1
  max_dose: int = 5
  waitlist_time: float = 0.0
  course_length: int = 28
  max_screen_duration: int = 28
  screen_fail_prob: float = 0.30
  ineval_prob: float = 0.20
  mean_interarrival_time: float = 10.0

  alpha_screening: float = 1
  beta_screening: float = 1
  alpha_dlt: float = 1.5
  beta_dlt: float = 1
  alpha_ineval: float = 1
  beta_ineval: float = 1

  scale: float = 0.2
  fifty_rate: float = 8.5

  # Custom DLT list
  dlt_probs: Optional[List[float]] = None

  # DLT Probability Function (defines the probability of a patient having a DLT event, based on dose)
  def get_dlt_probability(self, dose_level):
    if dose_level is None or dose_level <= 0:
      return 0.0

    # If a custom list is provided (e.g. [.1, .15, .25, .30, .35, .5])
    if self.dlt_probs is not None:
      idx = dose_level - 1
      # Return the value at the index, or the last value if dose exceeds list length
      if idx < len(self.dlt_probs):
        return self.dlt_probs[idx]
      else:
        return self.dlt_probs[-1]


    # Default: Arctan Formula
    val = 0.5 + (math.atan(self.scale * math.pi * (dose_level - self.fifty_rate)) / math.pi)
    return max(0.0, min(1.0, val))


  # Patient Inter-Arrival Time (days between arrivals)
  def sample_interarrival(self, rng):
    # return math.floor(rng.exponential(10)) # Mean of x days
    return rng.exponential(self.mean_interarrival_time)

  # Screening Duration Distribution (length of screening process)
  def sample_screening_duration(self, rng):
    # beta(0, 28, 1, 1) is a uniform distribution between 0 and 28
    return rng.beta(self.alpha_screening, self.beta_screening) * self.max_screen_duration

  # Time Until DLT Event (distribution of times before the patient will have the DLT event)
  def sample_dlt_timing(self, rng):
    # beta(0, CourseLength, 1.5, 1) skewed toward the end of the cycle
    return rng.beta(self.alpha_dlt, self.beta_dlt) * self.course_length

  # Time Until Inevaluable (distribution of times before the patient will become inevaluable)
  def sample_inevaluable_timing(self, rng):
    # beta(0,CourseLength,1,1) is a Uniform(0, 28) distribution
    return rng.beta(self.alpha_ineval, self.beta_ineval) * self.course_length

#Patient Class

In [5]:
class Patient:
  def __init__(self, arrival_time, dose, ID, rng, config: TrialConfig):
    self.ID = ID
    self.arrival_time = arrival_time
    self.dose = dose
    #self.course_length = course_length #length of study
    self.rng = rng
    #self.custom_dlt_probs = custom_dlt_probs
    self.config = config

    self.started_screening = False
    self.screening_start_time = None
    self.screening_end_time = None

    self.started_treatment = False
    self.treatment_start_time = None
    self.resolution_time = None
    self.treatment_outcome = None
    self.days_to_treatment_outcome = None

    self.screening_outcome, self.days_to_screening_outcome = self.determine_screening_outcome()

  def determine_screening_outcome(self):
    screen_duration = self.config.sample_screening_duration(self.rng)
    if self.rng.random() < self.config.screen_fail_prob:
      outcome = "fail"
    else:
      outcome = "pass"
    return outcome, screen_duration

  def start_screening(self, current_clock_time):
    self.started_screening = True
    self.screening_start_time = current_clock_time
    self.screening_end_time = self.screening_start_time + self.days_to_screening_outcome


  # returns "dlt", "ineval", or "pass", as well as a duration til it happens
  def determine_patient_outcome(self):
    dlt_prob = self.config.get_dlt_probability(self.dose)

    time_to_ineval = -1
    time_to_dlt = -1

    if self.rng.random() < self.config.ineval_prob:
      time_to_ineval = self.config.sample_inevaluable_timing(self.rng)

    if self.rng.random() < dlt_prob:
      time_to_dlt = self.config.sample_dlt_timing(self.rng)

    # if these both happen, we choose the one that will happen first in the timeline
    if time_to_ineval > -1 and time_to_dlt > -1:
      if time_to_ineval < time_to_dlt:
        return "ineval", time_to_ineval
      else:
        return "dlt", time_to_dlt
    elif time_to_ineval > -1:
      return "ineval", time_to_ineval
    elif time_to_dlt > -1:
      return "dlt", time_to_dlt
    else:
      return "pass", self.config.course_length

  # current_clock_time will be an int representation of days since trial
  # resolution_time is the day they pass, get dlt, or become ineval
  def start_treatment(self, current_clock_time):
    self.treatment_outcome, self.days_to_treatment_outcome = self.determine_patient_outcome()
    self.started_treatment = True
    self.treatment_start_time = current_clock_time
    self.resolution_time = self.treatment_start_time + self.days_to_treatment_outcome


#Simulation Code
- running this runs one simulation

## helpers

In [6]:
def archive(treated_list, pending_list):
  treated_list.extend(pending_list)
  pending_list.clear()

def get_decision_dict(trial_type):
  type_map = {
      1: decision_dict_33,
      2: decision_dict_IQ_33,
      3: decision_dict_6,
      4: decision_dict_IQ_6
  }
  return type_map.get(trial_type, None)

def remove_screening_from_dose(dose, screening_list, patient_states):
  # Removes the 'reservation' for screening patients from a dose level.
  for patient in screening_list:
      patient_states[dose]["treated"] -= 1
      patient_states[dose]["pending"] -= 1

def add_screening_to_dose(dose, screening_list, patient_states):
  # Assigns screening patients to a new dose level and updates counts.
  for patient in screening_list:
      patient.dose = dose
      patient_states[dose]["treated"] += 1
      patient_states[dose]["pending"] += 1

def finalize_pending_at_dose(dose, pending_list, patient_states):
  # Resolves outcomes for patients currently in treatment at a dose being exited.
  for patient in pending_list:
    patient_states[dose]["pending"] -= 1
    if patient.treatment_outcome == "dlt":
      patient_states[dose]["DLT"] += 1
    elif patient.treatment_outcome == "ineval":
      patient_states[dose]["treated"] -= 1


## event based sim

In [7]:
def simulation(trial_type, config: TrialConfig, seed=None, p_state = False):
  rng = np.random.default_rng(seed)
  decision_dict = get_decision_dict(trial_type)
  event_queue = []
  treated_list = []
  screening_list = []
  pending_list = []
  current_clock_time = 0.0

  ## Trial State
  trial_status = True
  # options are Stay, Up, Down, Suspend, MTD
  current_trial_event = "Stay"
  dose_level = config.starting_dose
  max_dose_level = config.max_dose
  min_dose_level = config.min_dose
  final_dose_level = None

  # for cohort expansion
  mtd_pending = []
  mtd_screening = []

  patientID = 1
  total_inevals = 0
  overlooked_patients = 0

  # Event Types
  ARRIVAL = 1
  SCREENING_END = 2
  TREATMENT_END = 3

  patient_states = {}
  for d in range(min_dose_level, max_dose_level + 1):
    patient_states[d] = {
      "treated": 0,
      "DLT": 0,
      "pending": 0,
      "AboveLevelState": "closed" if d == max_dose_level else "open"
    }

  # Helper to get the event_type
  def get_event():
    return decision_dict.get((
      patient_states[dose_level]['treated'],
      patient_states[dose_level]['DLT'],
      patient_states[dose_level]['pending'],
      patient_states[dose_level]['AboveLevelState']), "Error")

  # Start the trial with the first arrival
  first_arrival = config.sample_interarrival(rng)
  # heap called event queue has the arrival time, event type,
  heapq.heappush(event_queue, (first_arrival, ARRIVAL, None))

  while event_queue and trial_status:
    # jump the clock
    event_time, event_type, patient = heapq.heappop(event_queue)
    current_clock_time = event_time

    # patient arrives
    if event_type == ARRIVAL:
      # Schedule the next arrival regardless of current trial state
      state = patient_states[dose_level]
      next_arrival_time = current_clock_time + config.sample_interarrival(rng)
      heapq.heappush(event_queue, (next_arrival_time, ARRIVAL, None))

      if current_trial_event == "Stay":
        new_patient = Patient(current_clock_time, dose_level, patientID, rng, config)
        patientID += 1
        new_patient.start_screening(current_clock_time)
        screening_list.append(new_patient)
        heapq.heappush(event_queue, (new_patient.screening_end_time, SCREENING_END, new_patient))

        # If they are screening, they take up a spot in the study
        state["pending"] += 1
        state["treated"] += 1

      else:
        overlooked_patients += 1


    # patient finishes screening or inevals
    elif event_type == SCREENING_END:
      # Important: Check if patient is still in the screening_list (hasn't been cleared)
      state = patient_states[dose_level]
      if patient in screening_list:
        screening_list.remove(patient)
        if patient.screening_outcome == "fail":
          state["pending"] -= 1
          state["treated"] -= 1
        elif patient.screening_outcome == "pass":
          # Re-verify dose
          patient.dose = dose_level
          patient.start_treatment(current_clock_time)
          pending_list.append(patient)
          heapq.heappush(event_queue, (patient.resolution_time, TREATMENT_END, patient))

    # patient finishes treatment or dlt or inevals
    elif event_type == TREATMENT_END:
      state = patient_states[dose_level]
      if patient in pending_list:
        pending_list.remove(patient)
        state["pending"] -= 1
        if patient.treatment_outcome == "ineval":
          state["treated"] -= 1
          total_inevals += 1
        elif patient.treatment_outcome == "dlt":
          state["DLT"] += 1
        treated_list.append(patient)

    # update event type based on above factors
    current_trial_event = get_event()

    if current_trial_event == "Up":
      old_dose_level = dose_level
      dose_level += 1
      new_state = "closed" if dose_level == max_dose_level else "open"
      patient_states[dose_level]["AboveLevelState"] = new_state

      remove_screening_from_dose(old_dose_level, screening_list, patient_states)
      finalize_pending_at_dose(old_dose_level, pending_list, patient_states)
      add_screening_to_dose(dose_level, screening_list, patient_states)

      archive(treated_list, pending_list)

    elif current_trial_event == "Down":
      remove_screening_from_dose(dose_level, screening_list, patient_states)
      finalize_pending_at_dose(dose_level, pending_list, patient_states)

      while True:
        if dose_level == min_dose_level:
          final_dose_level = dose_level -1
          trial_status = False
          break

        dose_level -= 1
        patient_states[dose_level]["AboveLevelState"] = "closed"
        current_trial_event = get_event()

        if current_trial_event == "Down":
          # Level is still too toxic, down again
          continue
        elif current_trial_event == "MTD":
          final_dose_level = dose_level
          trial_status = False
          break
        else:
          # NOW we can safely add the screening patients
          if current_trial_event != "Error":
            add_screening_to_dose(dose_level, screening_list, patient_states)
            patient_needed = True
          else:
            trial_status = False
          break

      archive(treated_list, pending_list)

    # if current_trial_event == "Suspend" or current_trial_event == "Stay":
      # nothing happens here

    if current_trial_event == "MTD":
      if p_state:
        mtd_pending = pending_list[:]
        mtd_screening = screening_list[:]
      else:
        archive(treated_list, pending_list)
      final_dose_level = dose_level
      trial_status = False

    if current_trial_event == "Error":
      archive(treated_list, pending_list)
      final_dose_level = dose_level

      if dose_level == min_dose_level:
        result_summary = "Terminated: Too Toxic at Lowest Level"
      else:
        result_summary = "Terminated: Error/Inconclusive"

      print(f"Error in chart reached at seed {seed}")
      print(f"Reason: {result_summary}")
      end_on_error = True
      trial_status = False

    current_trial_event = get_event()

  if final_dose_level is None:
    print("trial timeout")
    final_dose_level = dose_level

  if p_state:
    return treated_list, current_clock_time, final_dose_level, total_inevals, patient_states, mtd_pending, mtd_screening

  return treated_list, current_clock_time, final_dose_level, total_inevals



#Debug code for simulation
- slightly outdated to the main code

## Helper methods

In [8]:
def debug_log(log_data_list, day, event, action, dose, states):

  # Create dictionary for CSV
  entry = {
    "Day": round(day, 3),
    "Event": event,
    "Action": action,
    "Dose": dose,
    "Treated": states.get('treated'),
    "DLT": states.get('DLT'),
    "Pending": states.get('pending'),
    "AboveState": states.get('AboveLevelState')
  }
  log_data_list.append(entry)

  return f"Day: {day:<7} | Event: {event:<10} | {action:<77} | Dose: {dose:<4} | States: {states} \n"

def get_decision_dict_debug(trial_type, event_log):
  if trial_type == 1:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_33
  elif trial_type == 2:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_IQ_33
  elif trial_type == 3:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_6
  elif trial_type == 4:
    event_log += "Beginning 3+3 Simulation\n"
    return decision_dict_IQ_6
  else:
    print("Invalid trial type: Please choose a value 1-4")
    return None

#Handles the state transfer for patients when the dose level changes
def migrate_patients_on_dose_change_debug_up(
    old_dose, new_dose, screening_list, pending_list, patient_states,
    current_clock_time, current_trial_event, log_data_list):
  # Migrate screening patients
  screen_string = ""
  event_log = ""
  for patient in screening_list:
    patient_states[old_dose]["treated"] -= 1
    patient_states[old_dose]["pending"] -= 1
    patient.dose = new_dose
    patient_states[new_dose]["treated"] += 1
    patient_states[new_dose]["pending"] += 1
    screen_string += f"{patient.ID}, "

  ids = [p.ID for p in pending_list]
  event_log += debug_log(log_data_list, round(current_clock_time, 2), current_trial_event, f"Patient {ids} archived. Dose goes up. Patients: {screen_string}migrated from screening", new_dose, patient_states[new_dose])

  # Resolve outcomes for those already in treatment
  for patient in pending_list:
    patient_states[old_dose]["pending"] -= 1
    event_log += debug_log(log_data_list, round(current_clock_time, 2), current_trial_event, f"Patient {patient.ID} will resolve to {patient.treatment_outcome} on day {patient.resolution_time}", new_dose, patient_states[new_dose])
    if patient.treatment_outcome == "dlt":
      patient_states[old_dose]["DLT"] += 1
    elif patient.treatment_outcome == "ineval":
      patient_states[old_dose]["treated"] -= 1

  return event_log


##Simulation (Discrete Event)

In [9]:
# New code
def simulation_with_debug(trial_type, config: TrialConfig, seed=None):
  # no actual seed on debug unless you want
  rng = None
  if seed == None:
    ss = np.random.SeedSequence(seed)
    actual_seed = ss.entropy
    #print(f"Running simulation with seed: {actual_seed}")
    rng = np.random.default_rng(ss)
    seed = actual_seed
  else:
    rng = np.random.default_rng(seed)

  # to keep track of patients
  decision_dict = get_decision_dict(trial_type)
  event_queue = []
  treated_list = []
  screening_list = []
  pending_list = []
  current_clock_time = 0.0

  # debug specifig variables
  event_log = f"Beginning Simulation (Type {trial_type}) with Seed {seed}\n"
  log_data_list = [] # for csv later
  end_on_error = False
  num_consented_patients = 0
  num_started_treatment = 0

  def get_day():
    return round(current_clock_time, 2)

  ## Trial State
  trial_status = True
  current_trial_event = "Stay"
  dose_level = config.starting_dose
  max_dose_level = config.max_dose
  min_dose_level = config.min_dose
  final_dose_level = None

  patientID = 1
  total_inevals = 0
  overlooked_patients = 0

  # Event Types
  ARRIVAL = 1
  SCREENING_END = 2
  TREATMENT_END = 3

  patient_states = {}
  for d in range(min_dose_level, max_dose_level + 1):
    patient_states[d] = {
      "treated": 0,
      "DLT": 0,
      "pending": 0,
      "AboveLevelState": "closed" if d == max_dose_level else "open"
    }

  # Helper to get the event_type
  def get_event():
    return decision_dict.get((
      patient_states[dose_level]['treated'],
      patient_states[dose_level]['DLT'],
      patient_states[dose_level]['pending'],
      patient_states[dose_level]['AboveLevelState']), "Error")

  # Start the trial with the first arrival
  first_arrival = config.sample_interarrival(rng)
  # heap called event queue has the arrival time, event type,
  heapq.heappush(event_queue, (first_arrival, ARRIVAL, None))

  while event_queue and trial_status:
    # jump the clock
    event_time, event_type, patient = heapq.heappop(event_queue)
    current_clock_time = event_time

    # patient arrives
    if event_type == ARRIVAL:
      # Schedule the next arrival regardless of current trial state
      state = patient_states[dose_level]
      next_arrival_time = current_clock_time + config.sample_interarrival(rng)
      heapq.heappush(event_queue, (next_arrival_time, ARRIVAL, None))

      if current_trial_event == "Stay":
        new_patient = Patient(current_clock_time, dose_level, patientID, rng, config)
        patientID += 1
        new_patient.start_screening(current_clock_time)
        screening_list.append(new_patient)
        num_consented_patients += 1
        heapq.heappush(event_queue, (new_patient.screening_end_time, SCREENING_END, new_patient))

        # If they are screening, they take up a spot in the study
        state["pending"] += 1
        state["treated"] += 1
        event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Adding new patient {new_patient.ID} to screening.", dose_level, patient_states[dose_level])

      else:
        overlooked_patients += 1
        event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Turned away possible patient", dose_level, patient_states[dose_level])

    # patient finishes screening or inevals
    elif event_type == SCREENING_END:
      # Important: Check if patient is still in the screening_list (hasn't been cleared)
      state = patient_states[dose_level]
      if patient in screening_list:
        screening_list.remove(patient)
        if patient.screening_outcome == "fail":
          state["pending"] -= 1
          state["treated"] -= 1
          event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Patient {patient.ID} failed screening.",dose_level, patient_states[dose_level])
        elif patient.screening_outcome == "pass":
          # Re-verify dose
          patient.dose = dose_level
          patient.start_treatment(current_clock_time)
          num_started_treatment += 1
          pending_list.append(patient)
          heapq.heappush(event_queue, (patient.resolution_time, TREATMENT_END, patient))
          event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Patient {patient.ID} starts treatment, at dose level {patient.dose}.", dose_level, patient_states[dose_level])

    # patient finishes treatment or dlt or inevals
    elif event_type == TREATMENT_END:
      state = patient_states[dose_level]
      if patient in pending_list:
        pending_list.remove(patient)
        state["pending"] -= 1
        if patient.treatment_outcome == "pass":
          event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Patient {patient.ID} passes treatment.", dose_level, state)
        if patient.treatment_outcome == "ineval":
          state["treated"] -= 1
          total_inevals += 1
          event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Removing patient {patient.ID} for ineval.", dose_level, state)
        elif patient.treatment_outcome == "dlt":
          state["DLT"] += 1
          event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Patient {patient.ID} has a DLT.", dose_level, state)
        treated_list.append(patient)

    # update event type based on above factors
    current_trial_event = get_event()

    if current_trial_event == "Up":
      old_dose_level = dose_level
      dose_level += 1
      new_state = "closed" if dose_level == max_dose_level else "open"
      patient_states[dose_level]["AboveLevelState"] = new_state

      #migrating the screening patients to the next dose
      event_log += migrate_patients_on_dose_change_debug_up(
        old_dose_level, dose_level, screening_list, pending_list, patient_states,
        current_clock_time, current_trial_event, log_data_list)

      archive(treated_list, pending_list)

    elif current_trial_event == "Down":
      remove_screening_from_dose(dose_level, screening_list, patient_states)
      finalize_pending_at_dose(dose_level, pending_list, patient_states)

      p_ids = [p.ID for p in pending_list]
      s_ids = [p.ID for p in screening_list]

      while True:
        if dose_level == min_dose_level:
          final_dose_level = dose_level -1
          trial_status = False
          break

        dose_level -= 1
        patient_states[dose_level]["AboveLevelState"] = "closed"
        current_trial_event = get_event()

        if current_trial_event == "Down":
          # Level is still too toxic, down again
          continue
        elif current_trial_event == "MTD":
          final_dose_level = dose_level
          trial_status = False
          event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Decreasing Dose. Moving patients {p_ids} to treated from pending, dose is now MTD.", dose_level, patient_states[dose_level])
          break
        else:
          # NOW we can safely add the screening patients
          if current_trial_event != "Error":
            add_screening_to_dose(dose_level, screening_list, patient_states)
            patient_needed = True
          else:
            trial_status = False
          break

      event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Patient {p_ids} archived and {s_ids} transferred. Dose goes down to dose level {dose_level}.", dose_level, patient_states[dose_level])
      archive(treated_list, pending_list)

    # if current_trial_event == "Suspend" or current_trial_event == "Stay":
      # nothing happens here

    if current_trial_event == "MTD":
      ids = [p.ID for p in pending_list]
      event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Moving patients {ids} to treated from pending, dose is MTD.", dose_level, patient_states[dose_level])

      archive(treated_list, pending_list)

      final_dose_level = dose_level
      trial_status = False

    if current_trial_event == "Error":
      ids = [p.ID for p in pending_list]
      event_log += debug_log(log_data_list, get_day(), current_trial_event, f"Moving patients {ids} to treated from pending, error.", dose_level, patient_states[dose_level])

      archive(treated_list, pending_list)
      final_dose_level = dose_level

      if dose_level == min_dose_level:
        result_summary = "Terminated: Too Toxic at Lowest Level"
      else:
        result_summary = "Terminated: Error/Inconclusive"

      print(f"Error in chart reached at seed {seed}")
      print(f"Reason: {result_summary}")
      end_on_error = True
      trial_status = False

    current_trial_event = get_event()

  if final_dose_level is None:
    print("trial timeout")
    final_dose_level = dose_level

  num_fully_evaluable = len([p for p in treated_list if p.treatment_outcome != "ineval"])
  event_log += f"Number of patients consented to screening: {num_consented_patients}\n"
  event_log += f"Number of patients starting treatment: {num_started_treatment}\n"
  event_log += f"Number of patients fully evaluable: {num_fully_evaluable}\n"
  event_log += f"Final Dose Level: {final_dose_level}\n"
  event_log += f"Total Time Taken: {round(current_clock_time, 3)} days or ~{round(current_clock_time / 30.375 , 2)} months\n"
  # print(f"Overlooked patients: {overlooked_patients}")

  # returns list of all treated/dlt patients, days_taken, final_dose_level, total_inevals
  return treated_list, current_clock_time, final_dose_level, total_inevals, event_log, end_on_error, seed, log_data_list



## For Debug Output

In [10]:
def run_debug_test(scenario=None, sim_type=1, seed=None, save_csv=True, **kwargs):
  base_config = TrialConfig()

  # If a scenario dict is passed (e.g., from your supplimental_scenarios list)
  if scenario is not None:
    # Use your cleaning logic to filter out "name" and other non-config keys
    valid_fields = {k: v for k, v in scenario.items() if hasattr(base_config, k)}
    config = replace(base_config, **valid_fields)
    scenario_name = scenario.get("name", "Scenario_Run")

  # Otherwise, use the individual keyword arguments
  else:
    # We still clean kwargs just in case you pass something like name="test"
    valid_fields = {k: v for k, v in kwargs.items() if hasattr(base_config, k)}
    config = replace(base_config, **valid_fields)
    scenario_name = "Default_Run"

  if seed is None:
    seed = np.random.SeedSequence().entropy

  print(f"--- DEBUG START: {scenario_name} | Seed: {seed} ---")

  # Run the simulation
  treated, time, mtd, inevals, log, err, seed, log_data = simulation_with_debug(sim_type, config, seed)

  # Print the console log
  print(log)

  # Save CSV
  if save_csv:
    df_log = pd.DataFrame(log_data)
    clean_name = scenario_name.replace("\n", "_").replace(" ", "_")
    filename = f"debug_{clean_name}.csv"
    df_log.to_csv(filename, index=False)
    print(f"--- Data saved to {filename} ---")

In [11]:
def run_debug_test_infinite(sim_type = 1, reps = 10000, seed = None, **kwargs):
  base_config = TrialConfig()
  config = replace(base_config, **kwargs)

  if seed is None:
    seed = np.random.SeedSequence().entropy

  for i in range(reps):
    result = simulation_with_debug(sim_type, config, seed)
    if result[5]:
      print(f"Seed: {result[6]}\n {result[4]}")
      break

#Metrics
- This runs x num of simulations and returns the metrics
- Used directly in the run_batch_report dunction

In [12]:
def get_metrics(trial_type: int, num_repetitions: int, config: TrialConfig, start_seed: int):
  metrics = []
  # Track aggregate dose stats across all trials
  dose_stats = defaultdict(lambda: {"dlts": 0, "total": 0})

  for i in range(num_repetitions):
    treated_list, days_taken, final_dose, total_inevals = simulation(trial_type, config, seed = start_seed + i)

    # Identify Evaluable patients (outcome is not 'ineval')
    evaluable_list = [p for p in treated_list if p.treatment_outcome != "ineval"]

    num_DLT_above_MTD = 0
    for p in treated_list:
      dose_stats[p.dose]["total"] += 1
      if p.treatment_outcome == "dlt":
        dose_stats[p.dose]["dlts"] += 1
        if final_dose is not None and p.dose > final_dose:
          num_DLT_above_MTD += 1

    metrics.append({
      "days_taken": days_taken,
      "final_dose": final_dose,
      "num_treated": len(treated_list),      # Total count that started
      "num_evaluable": len(evaluable_list),  # Completed DLT window
      "num_DLT_above_MTD": num_DLT_above_MTD
    })

  return pd.DataFrame(metrics), dose_stats

def format_theoretical_dlt_col(config: TrialConfig):
  # Formats the 'Ground Truth' column based on either custom list or mathematical model.
  lines = ["Dose  Prob"]
  for level in range(1, config.max_dose + 1):
    prob = config.get_dlt_probability(level)
    lines.append(f"{level:<4}  {prob*100:>4.1f}%")
  return "\n".join(lines)

def format_stat_str(ser: pd.Series):
  # Returns 'Mean, Med (Min-Max)'
  m, med = ser.mean(), ser.median()
  mi, ma = ser.min(), ser.max()
  return f"{m:.1f}, {med:.1f}\n({int(mi)}-{int(ma)})"

def format_mtd_distribution(doses: pd.Series, config: TrialConfig):
  # Calculates how often each dose was selected as MTD
  # Handle trials that failed (MTD = 0 or None)
  counts = doses.fillna(0).value_counts(normalize=True) * 100
  lines = []
  # Check levels 0 through Max
  for lvl in range(0, config.max_dose + 1):
      pct = counts.get(lvl, 0.0)
      label = "Fail" if lvl == 0 else f"DL {lvl}"
      lines.append(f"{label}: {pct:>5.1f}%")
  return "\n".join(lines)


In [13]:
def run_batch_report(scenario_defs, sim_types, num_reps, seed=0, show_evaluable=False):
  type_map = {1: "3+3", 2: "IQ 3+3", 3: "Roll 6", 4: "IQ Roll 6"}
  table_rows = []
  base_config = TrialConfig()

  for scenario in scenario_defs:
    # Clean the scenario dict to only use valid config fields
    valid_fields = {k: v for k, v in scenario.items() if hasattr(base_config, k)}
    config = replace(base_config, **valid_fields)

    row = {"Variation": scenario.get("name", "Unnamed")}

    # Horizontal Grouping Dictionaries
    treated_cols = {}
    eval_cols = {}
    month_cols = {}
    mtd_cols = {}
    safety_summary = []

    # Run simulations and collect data
    for st in sim_types:
      df, _ = get_metrics(st, num_reps, config, seed)
      df['months'] = df['days_taken'] / 30.4375
      lbl = type_map[st]

      # Always collect Treated and Months
      treated_cols[f"#Treated\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['num_treated'])
      month_cols[f"#Months\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['months'])

      # Only collect Evaluable if requested
      if show_evaluable:
        eval_cols[f"#Evaluable\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['num_evaluable'])

      mtd_cols[f"{lbl}\n% MTD\nAt level"] = format_mtd_distribution(df['final_dose'], config)
      safety_summary.append(f"{lbl}: {df['num_DLT_above_MTD'].mean():.2f}")

    # Assemble rows
    row.update(treated_cols)

    if show_evaluable:
      row.update(eval_cols)

    row.update(month_cols)
    row["Mean\n#DLTs\nabove\nMTD"] = "\n".join(safety_summary)
    row.update(mtd_cols)
    row["DLT rate\n(Ground\nTruth)"] = format_theoretical_dlt_col(config)

    table_rows.append(row)

  print(tabulate(table_rows, headers="keys", tablefmt="grid"))

#Supplemental Scenarios

In [14]:
supplemental_scenarios = [
    {
      "name": "Scenario A1\nAve Ineval\nPhase 1\n20% Ineval"
    },
    {
      "name": "Scenario A2\nLow Ineval\nPhase 1\n3.6% Ineval",
      "ineval_prob": 0.036
    },
    {
      "name": "Scenario A3\nHigh Ineval\nPhase 1\n44% Ineval",
      "ineval_prob": 0.44
    },
    {
      "name": "Scenario A4\nAve Ineval\nPhase 1\nScreenfail to 60%",
      "screen_fail_prob": 0.6
    },
    {
      "name": "Scenario A5\nAve Ineval\nPhase 1\nArrival (15 day mean)",
      "mean_interarrival_time": 15.0
    },
    {
      "name": "Scenario A6\nAve Ineval\nPhase 1\nCL to 21 Days",
      "course_length": 21
    },
    {
      "name": "Scenario A7\nAve Ineval\nPhase 1\nToxicity",
      # "dlt_probs": [0.108, 0.136, 0.18, 0.259, 0.403],
      "fifty_rate": 5.5
    },
    {
      "name": "Scenario B\nSafety Lead-in\n Phase 1",
      "max_dose": 2,
      # "dlt_probs": [0.108, 0.136]
      "fifty_rate": 5.5,
      "course_length": 21
    },
    # C1, C2, C3
    # {
    #   "name": "Scenario C1\n2nd cycle with\n experimental combo",
    #   "max_dose": 4,
    #   "dlt_probs": [0.059, 0.067, 0.076, 0.090]
    # },
    # {
    #   "name": "Scenario C2\n2nd cycle \n Arrival to 15",
    #   "max_dose": 4,
    #   "dlt_probs": [0.059, 0.067, 0.076, 0.090],
    #   "interarrival_time": 15.0
    # },
    # {
    #   "name": "Scenario C3\n2nd cycle \n Ineval reate to 33%",
    #   "ineval_prob": 0.33,
    #   "max_dose": 4,
    #   "dlt_probs": [0.059, 0.067, 0.076, 0.090]
    # },
    {
      "name": "Scenario D\nIP Phase 1",
      # "dlt_probs": [0.053, 0.059, 0.067, 0.076, 0.090, 0.108]
      "max_dose": 6,
      "starting_dose": 1,
      "fifty_rate": 10.5,
      "max_screen_duration": 90,
      "beta_screening": 1.97,
      "screen_fail_prob": 0.40,
      "ineval_prob": 0.075,
      "mean_interarrival_time": 15

    }
]





##eTable 3

In [15]:
print("eTable 3 ::: Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)")
run_batch_report(supplemental_scenarios, [1, 2], 800)

eTable 3 ::: Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)
+-----------------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+-------------+
| Variation             | #Treated    | #Treated    | #Months     | #Months     | Mean         | 3+3          | IQ 3+3       | DLT rate    |
|                       | 3+3         | IQ 3+3      | 3+3         | IQ 3+3      | #DLTs        | % MTD        | % MTD        | (Ground     |
|                       | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above        | At level     | At level     | Truth)      |
|                       | (range)     | (range)     | (range)     | (range)     | MTD          |              |              |             |
+=======================+=============+=============+=============+=============+==============+==============+==============+=============+
| Scenario A1           | 19.0, 19.0  | 21.7, 22.0  | 19.4, 19.3  | 15.6, 15.5

##eTable 4

In [16]:
print("eTable 4 ::: Simulations for R6 and IQ R6 (each based on 800 simulations)")
run_batch_report(supplemental_scenarios, [3, 4], 800)

eTable 4 ::: Simulations for R6 and IQ R6 (each based on 800 simulations)
+-----------------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------------+-------------+
| Variation             | #Treated    | #Treated    | #Months     | #Months     | Mean            | Roll 6       | IQ Roll 6    | DLT rate    |
|                       | Roll 6      | IQ Roll 6   | Roll 6      | IQ Roll 6   | #DLTs           | % MTD        | % MTD        | (Ground     |
|                       | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above           | At level     | At level     | Truth)      |
|                       | (range)     | (range)     | (range)     | (range)     | MTD             |              |              |             |
+=======================+=============+=============+=============+=============+=================+==============+==============+=============+
| Scenario A1           | 23.6, 25.5  | 23.2, 24.0  | 15.9, 16



This code block runs simulations and generates a report formatted to match the paper's supplemental **eTable 3**.

## How to customize scenarios
You can add multiple rows to the table by adding dictionaries to the `scenarios_to_run` list. For each scenario, you can define a `name` and any of the following global variables to override:

*   `name` (No Default) Name your simulation.
*   `starting_dose` (Default: 2)
*   `min_dose` (Default: 1)
*   `max_dose` (Default: 5)
*   `waitlist_time` (Default: 0)
*   `course_length` (Default: 28)
*   `max_screen_duration`: (Default: 28)
*   `screen_failure_prob` (Default: 0.30)
*   `ineval_prob` (Default: 0.20)
*   `mean_interarrival_time` (Default: 10.0)
*   `dlt_probs` (defaults to arctan formula)
    * To use, provide a list that contains values for every dose level desired.
    * Examlple:  "dlt_probs": [0.10, 0.15, 0.25, 0.30, 0.35, 0.50] for 6 dose levels.

*   `alpha_screening` (Default: 1)
*   `beta_screening` (Default: 1)
*   `alpha_dlt` (Default: 1.5)
*   `beta_dlt` (Default: 1)
*   `alpha_ineval` (Default: 1)
*   `beta_ineval` (Default: 1)

### Function Parameters
1. **`scenarios`**: The list of dictionaries defined above.
2. **`sim_types`**: The simulation IDs to run side-by-side (e.g., `[1, 2]` for 3+3 and IQ 3+3).
3. **`num_repetitions`**: The number of simulated trials per scenario (e.g., 800).
4. **`seed`**: Change to random integer to change the patient batch. (default: 0).
5. **`show_evaluable`**: (Default: False) toggle true to display num evaluable patients per study.

### Example
```python
scenario_1 = [
    {
        "name": "Scenario A1\nAve Ineval\n20% Ineval",
        "ineval_prob": 0.20,
        "starting_dose": 2
    },
    { another scenario }
]

run_batch_report(scenarios_to_run, [1, 2], 8000, seed = 32498324865654)
```

###Example 1

In [17]:
# Define your scenarios here.
# Anything not defined in the dict will use the current global default.
scenarios_to_run = [
    {
        "name": "Scenario A1\nAve Ineval\n20% Ineval",
        "ineval_prob": 0.20,
        "starting_dose": 2
    },
    {
        "name": "Scenario B1\nAve Ineval\n20% Ineval",
        "ineval_prob": 0.20,
        "starting_dose": 2,
        "dlt_probs": [0.10, 0.15, 0.25, 0.30, 0.35]
    },
    {
        "name": "Scenario A2\nLow Ineval\n3.6% Ineval",
        "ineval_prob": 0.036,
        "starting_dose": 2
    },
    {
        "name": "Scenario B2\nLow Ineval\n3.6% Ineval",
        "ineval_prob": 0.036,
        "starting_dose": 2,
        "dlt_probs": [0.10, 0.15, 0.25, 0.30, 0.35, 0.50]
    }
]

print("Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)")
run_batch_report(scenarios_to_run, [1, 2], 8000, seed = 32498324865654)

Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)
+-------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+-------------+
| Variation   | #Treated    | #Treated    | #Months     | #Months     | Mean         | 3+3          | IQ 3+3       | DLT rate    |
|             | 3+3         | IQ 3+3      | 3+3         | IQ 3+3      | #DLTs        | % MTD        | % MTD        | (Ground     |
|             | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above        | At level     | At level     | Truth)      |
|             | (range)     | (range)     | (range)     | (range)     | MTD          |              |              |             |
+=============+=============+=============+=============+=============+==============+==============+==============+=============+
| Scenario A1 | 19.0, 19.0  | 21.8, 22.0  | 19.2, 19.2  | 15.7, 15.7  | 3+3: 0.89    | Fail:   0.5% | Fail:   0.7% | Dose  Prob  |
| Ave Ineval  | (6-3

In [18]:
scenarios_to_run = [
    {
        "name": "Example \nScenario 1\n(ineval:0)",
        "ineval_prob": 0,
        "screen_fail_prob": 0,
        "starting_dose": 1,
        "max_dose": 1,
        "min_dose": 1
    },
    {
        "name": "Example \nScenario 2\n(ineval:0.5)",
        "ineval_prob": 0.5,
        "screen_fail_prob": 0,
        "starting_dose": 1,
        "max_dose": 1,
        "min_dose": 1
    }

]

print("Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)")
run_batch_report(scenarios_to_run, [1, 2], 800)

Simulations for 3+3 and IQ 3+3 (each based on 800 simulations)
+--------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+-------------+
| Variation    | #Treated    | #Treated    | #Months     | #Months     | Mean         | 3+3          | IQ 3+3       | DLT rate    |
|              | 3+3         | IQ 3+3      | 3+3         | IQ 3+3      | #DLTs        | % MTD        | % MTD        | (Ground     |
|              | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above        | At level     | At level     | Truth)      |
|              | (range)     | (range)     | (range)     | (range)     | MTD          |              |              |             |
+==============+=============+=============+=============+=============+==============+==============+==============+=============+
| Example      | 5.9, 6.0    | 6.8, 7.0    | 4.6, 4.5    | 4.0, 3.9    | 3+3: 0.11    | Fail:   5.1% | Fail:   5.9% | Dose  Prob  |
| Scenario 1 

# Simulation Benchmarking (eTable 3 Comparison)

# Individual Simulation Event Log

Use this block to run a **single trial** and view the detailed step-by-step event logs. This is useful for debugging patient flow, dose escalations, and timing logic.

### How to use:
Call `run_debug_test()` and type the values you want to change. If you don't type a variable, it uses the default.

### Function Parameters
* `scenario`: Uses the same variables from the benchmarking  
* `sim_type`: [1-4]
* `seed`: (Default: Random) Enter an int value to specify seed.
* `save_csv`: (Default: True) Change to `False` if you do not want a new file every generation.

Files will save to the name given in the scenario

Find saved files on the left <-

Use the three dots to download or open on the right

Files created will be deleted on reset of runtime, be sure to download them if you want a permanent copy

### Sample Generation
```python
run_debug_test_2(sim_type=1)
```

```
Running simulation with seed: 123456
Beginning 3+3 Simulation
Day: 8    | Event: Stay       | Adding new patient 1 to screening.                                | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'}
...
Day: 1073 | Event: MTD        | Moving patients [] to treated from pending, dose is MTD.          | States: {'treated': 6, 'DLT': 1, 'pending': 0, 'AboveLevelState': 'closed'}
Number of patients consented to screening: 42
Number of patients starting treatment: 31
Number of patients fully evaluable: 27
Final Dose Level: 4
Total Time Taken: 1074 days or ~35.36 months
```



###Run
- If you come across a case that is incorrect in the future you can now refer to it as the `seed` that it uses, which should be a lot easier to descriabe to each other.
- even if you leave `seed` blank, it will tell you what seed was used.

In [19]:
# here is an example of a trial that goes up and then back down
run_debug_test(sim_type=2, seed = 301486315800757557521424412977664080381)


--- DEBUG START: Default_Run | Seed: 301486315800757557521424412977664080381 ---
Beginning Simulation (Type 2) with Seed 301486315800757557521424412977664080381
Day: 28.5    | Event: Stay       | Adding new patient 1 to screening.                                            | Dose: 2    | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'} 
Day: 32.48   | Event: Stay       | Adding new patient 2 to screening.                                            | Dose: 2    | States: {'treated': 2, 'DLT': 0, 'pending': 2, 'AboveLevelState': 'open'} 
Day: 32.5    | Event: Stay       | Adding new patient 3 to screening.                                            | Dose: 2    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 33.06   | Event: Suspend    | Turned away possible patient                                                  | Dose: 2    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 35.06   | Event: Suspe

In [20]:
Scenario_D = {
      "name": "Scenario D\nIP Phase 1",
      "max_dose": 6,
      "starting_dose": 1,
      "fifty_rate": 10.5,
      "max_screen_duration": 90,
      "beta_screening": 1.97,
      "screen_fail_prob": 0.40,
      "ineval_prob": 0.075,
      "mean_interarrival_time": 15
    }

In [21]:
run_debug_test(Scenario_D, sim_type= 1)

--- DEBUG START: Scenario D
IP Phase 1 | Seed: 51753942722315089279401019162351455796 ---
Beginning Simulation (Type 1) with Seed 51753942722315089279401019162351455796
Day: 18.9    | Event: Stay       | Adding new patient 1 to screening.                                            | Dose: 1    | States: {'treated': 1, 'DLT': 0, 'pending': 1, 'AboveLevelState': 'open'} 
Day: 20.65   | Event: Stay       | Adding new patient 2 to screening.                                            | Dose: 1    | States: {'treated': 2, 'DLT': 0, 'pending': 2, 'AboveLevelState': 'open'} 
Day: 48.87   | Event: Stay       | Adding new patient 3 to screening.                                            | Dose: 1    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 52.8    | Event: Suspend    | Turned away possible patient                                                  | Dose: 1    | States: {'treated': 3, 'DLT': 0, 'pending': 3, 'AboveLevelState': 'open'} 
Day: 52.9    | Even

# Cohort Expansion

In [22]:
import json

#strictly for visualization
def cohort_expansion_print(trial_type, expansion_num, config: TrialConfig, seed=None):
  treated_list, days_taken, final_dose, total_inevals, patient_states, pending, screening = simulation(trial_type, config, p_state=True)
  print(f"Trial Type: {trial_type}")
  print(f"Final Dose: {final_dose}")
  print(f"Patient States (cumulative after expansion):\n{json.dumps(patient_states, indent=2)}")
  print("Extra Patients (from initial MTD declaration):")
  if pending or screening:
    for patient in pending:
      print(f"  Patient ID: {patient.ID}, Dose Level: {patient.dose}, Started Treatment: {patient.started_treatment}")
    for patient in screening:
      print(f"  Patient ID: {patient.ID}, Dose Level: {patient.dose}, Started Treatment: {patient.started_treatment}")
  else:
    print("  No extra patients.")



cohort_expansion_print(2, 14, TrialConfig())


Trial Type: 2
Final Dose: 5
Patient States (cumulative after expansion):
{
  "1": {
    "treated": 0,
    "DLT": 0,
    "pending": 0,
    "AboveLevelState": "open"
  },
  "2": {
    "treated": 4,
    "DLT": 0,
    "pending": 0,
    "AboveLevelState": "open"
  },
  "3": {
    "treated": 3,
    "DLT": 0,
    "pending": 0,
    "AboveLevelState": "open"
  },
  "4": {
    "treated": 4,
    "DLT": 0,
    "pending": 0,
    "AboveLevelState": "open"
  },
  "5": {
    "treated": 8,
    "DLT": 1,
    "pending": 2,
    "AboveLevelState": "closed"
  }
}
Extra Patients (from initial MTD declaration):
  Patient ID: 24, Dose Level: 5, Started Treatment: True
  Patient ID: 25, Dose Level: 5, Started Treatment: True


In [23]:
def cohort_expansion(trial_type, expansion_num, config: TrialConfig, seed=None, replace_ineval=False, expand_levels_below_mtd=0):
  # 1. Unpack Phase 1 results
  treated_list, phase1_time, final_dose, total_inevals, patient_states, phase1_pending, phase1_screening = simulation(
      trial_type, config, seed=seed, p_state=True
  )

  # 2. Short-circuit for exact Phase 1 Baseline matching (or failure)
  if final_dose is None or final_dose < config.min_dose or expansion_num == 0:
    treated_list.extend(phase1_pending)
    return treated_list, phase1_time, phase1_time, final_dose, total_inevals, patient_states

  rng_seed = seed + 100000 if seed is not None else None
  rng = np.random.default_rng(rng_seed)

  event_queue = []
  ARRIVAL = 1
  SCREENING_END = 2
  TREATMENT_END = 3

  screening_list = []
  pending_list = []

  lowest_target_dose = max(config.min_dose, final_dose - expand_levels_below_mtd)
  target_doses = list(range(lowest_target_dose, final_dose + 1))

  # Bring over pending patients from Phase 1
  if phase1_pending:
    for p in phase1_pending:
      pending_list.append(p)
      heapq.heappush(event_queue, (p.resolution_time, TREATMENT_END, p))

  # --- PIPELINE STATE (For Accrual) ---
  valid_slots_locked = {d: 0 for d in target_doses}
  in_screening = {d: 0 for d in target_doses}
  future_arrival_scheduled = False

  # --- COMPLETION STATE (For the Trial Clock) ---
  completed_targets = {d: 0 for d in target_doses}

  for d in target_doses:
      if replace_ineval:
          valid_finished = len([p for p in treated_list if p.dose == d and p.treatment_outcome != "ineval"])
      else:
          valid_finished = len([p for p in treated_list if p.dose == d])

      # For Accrual: Pending patients count as 'locked' slots for now
      valid_pending = len([p for p in phase1_pending if p.dose == d])
      valid_slots_locked[d] = valid_finished + valid_pending

      # For Clock: Only people who actually finished Phase 1 count
      completed_targets[d] = valid_finished

  # Check if the "weird case" happened where Phase 1 already hit the target
  target_met = all(completed_targets[d] >= expansion_num for d in target_doses)
  official_end_time = phase1_time if target_met else None

  # Bring over screening patients from Phase 1
  if phase1_screening:
    for p in phase1_screening:
      if p.dose not in target_doses:
          p.dose = final_dose
      in_screening[p.dose] += 1
      screening_list.append(p)
      heapq.heappush(event_queue, (p.screening_end_time, SCREENING_END, p))

  patientID = max([p.ID for p in treated_list + phase1_pending + phase1_screening], default=0) + 1
  current_clock_time = phase1_time

  def get_best_dose_for_arrival():
      best_dose = None
      max_deficit = 0
      for d in reversed(target_doses):
          deficit = expansion_num - (valid_slots_locked[d] + in_screening[d])
          if deficit > max_deficit:
              max_deficit = deficit
              best_dose = d
      return best_dose

  def check_and_schedule_arrival():
      nonlocal future_arrival_scheduled
      if not future_arrival_scheduled and get_best_dose_for_arrival() is not None:
          next_arrival_time = current_clock_time + config.sample_interarrival(rng)
          heapq.heappush(event_queue, (next_arrival_time, ARRIVAL, None))
          future_arrival_scheduled = True

  # Check pipeline at the very start
  check_and_schedule_arrival()

  while event_queue:
    event_time, event_type, patient = heapq.heappop(event_queue)
    current_clock_time = max(current_clock_time, event_time)

    if event_type == ARRIVAL:
      future_arrival_scheduled = False
      target_dose = get_best_dose_for_arrival()

      if target_dose is not None:
          new_patient = Patient(current_clock_time, target_dose, patientID, rng, config)
          new_patient.is_expansion = True
          patientID += 1
          new_patient.start_screening(current_clock_time)
          screening_list.append(new_patient)
          heapq.heappush(event_queue, (new_patient.screening_end_time, SCREENING_END, new_patient))

          in_screening[target_dose] += 1
          patient_states[target_dose]["pending"] += 1
          patient_states[target_dose]["treated"] += 1

      check_and_schedule_arrival()

    elif event_type == SCREENING_END:
      if patient in screening_list:
        screening_list.remove(patient)
        d = patient.dose
        in_screening[d] -= 1

        if patient.screening_outcome == "fail":
          patient_states[d]["pending"] -= 1
          patient_states[d]["treated"] -= 1
          check_and_schedule_arrival()

        elif patient.screening_outcome == "pass":
          patient.start_treatment(current_clock_time)
          pending_list.append(patient)
          heapq.heappush(event_queue, (patient.resolution_time, TREATMENT_END, patient))
          valid_slots_locked[d] += 1 # Slot locked!
          check_and_schedule_arrival()

    elif event_type == TREATMENT_END:
      if patient in pending_list:
        pending_list.remove(patient)
        d = patient.dose
        patient_states[d]["pending"] -= 1

        if patient.treatment_outcome == "ineval":
          patient_states[d]["treated"] -= 1
          total_inevals += 1

          if replace_ineval and d in target_doses:
              valid_slots_locked[d] -= 1 # Lose locked slot
              check_and_schedule_arrival()
          elif not replace_ineval and d in target_doses:
              completed_targets[d] += 1 # It counts as a finished cycle

        elif patient.treatment_outcome in ["pass", "dlt"]:
          if patient.treatment_outcome == "dlt":
            patient_states[d]["DLT"] += 1
          if d in target_doses:
            completed_targets[d] += 1 # It counts as a finished cycle

        treated_list.append(patient)

        # CLOCK CHECK: If this exact event pushes us to our target, freeze the study time!
        if official_end_time is None and all(completed_targets[td] >= expansion_num for td in target_doses):
            official_end_time = current_clock_time

  # Fallback in case event queue empties (due to massive screen fails) and target is strictly unreachable
  if official_end_time is None:
      official_end_time = current_clock_time

  # Return the frozen `official_end_time` for exact study duration.
  return treated_list, official_end_time, phase1_time, final_dose, total_inevals, patient_states

##Cohort Metrics

In [24]:
def run_single_expansion(trial_type, expansion_num, replace_ineval=False, expand_levels_below_mtd=0, config=None, seed=None):
  if config is None:
    config = TrialConfig()
  if seed is None:
    seed = np.random.SeedSequence().entropy

  type_map = {1: "3+3", 2: "IQ 3+3", 3: "Roll 6", 4: "IQ Roll 6"}
  trial_name = type_map.get(trial_type, "Unknown")
  target_type_str = "Evaluable" if replace_ineval else "Started Treatment"

  print(f"--- Single Trial Metrics: {trial_name} | Target: {expansion_num}/DL ({target_type_str}) | Multi-DL: {expand_levels_below_mtd} levels below | Seed: {seed} ---")

  treated_list, total_time, phase1_time, final_dose, total_inevals, patient_states = cohort_expansion(
      trial_type, expansion_num, config, seed=seed, replace_ineval=replace_ineval, expand_levels_below_mtd=expand_levels_below_mtd
  )

  if final_dose is None or final_dose < config.min_dose:
    print(f"Result: Trial Terminated Early (Too toxic or error).")
    print(f"Final Dose Level Reached: {final_dose}")
    return

  lowest_target = max(config.min_dose, final_dose - expand_levels_below_mtd)
  target_doses = list(range(lowest_target, final_dose + 1))

  print(f"Result: MTD Declared at Dose Level {final_dose}")
  print(f"Expansion Cohorts Activated: Dose Levels {target_doses}\n")

  p1 = [p for p in treated_list if not getattr(p, 'is_expansion', False)]
  p2 = [p for p in treated_list if getattr(p, 'is_expansion', False)]

  def get_counts(plist, dose_filter=None):
    sublist = [p for p in plist if (dose_filter is None or p.dose == dose_filter)]
    evaluable = [p for p in sublist if p.treatment_outcome != "ineval"]
    inevals = [p for p in sublist if p.treatment_outcome == "ineval"]
    dlts = [p for p in sublist if p.treatment_outcome == "dlt"]
    return len(sublist), len(evaluable), len(inevals), len(dlts)

  p1_treat, p1_eval, p1_ineval, p1_dlts = get_counts(p1)
  p2_treat, p2_eval, p2_ineval, p2_dlts = get_counts(p2)
  tot_treat, tot_eval, tot_ineval, tot_dlts = get_counts(treated_list)

  phase2_time = total_time - phase1_time

  print("[ Time Taken ]")
  print(f"Phase 1 (To MTD) : {phase1_time:>6.1f} days (~{phase1_time/30.4375:.1f} months)")
  print(f"Phase 2 Expansion: {phase2_time:>6.1f} days (~{phase2_time/30.4375:.1f} months)")
  print(f"Total Time       : {total_time:>6.1f} days (~{total_time/30.4375:.1f} months)")

  print("\n[ Entire Trial (All Levels) Patient Counts ]")
  print(f"{'':<18} | {'Phase 1':<8} | {'Phase 2':<8} | {'Total':<8}")
  print("-" * 50)
  print(f"{'Started Treatment':<18} | {p1_treat:<8} | {p2_treat:<8} | {tot_treat:<8}")
  print(f"{'Evaluable':<18} | {p1_eval:<8} | {p2_eval:<8} | {tot_eval:<8}")
  print(f"{'Inevaluable':<18} | {p1_ineval:<8} | {p2_ineval:<8} | {tot_ineval:<8}")
  print(f"{'Total DLTs':<18} | {p1_dlts:<8} | {p2_dlts:<8} | {tot_dlts:<8}")

  # Print specific tables for each targeted dose level
  for d in reversed(target_doses):
    dp1_treat, dp1_eval, dp1_ineval, dp1_dlts = get_counts(p1, d)
    dp2_treat, dp2_eval, dp2_ineval, dp2_dlts = get_counts(p2, d)
    dtot_treat, dtot_eval, dtot_ineval, dtot_dlts = get_counts(treated_list, d)

    label_suffix = " (MTD)" if d == final_dose else ""
    print(f"\n[ Dose Level {d}{label_suffix} Specifics ]")
    print(f"{'':<18} | {'Phase 1':<8} | {'Phase 2':<8} | {'Total':<8}")
    print("-" * 50)
    print(f"{'Started Treatment':<18} | {dp1_treat:<8} | {dp2_treat:<8} | {dtot_treat:<8}")
    print(f"{'Evaluable':<18} | {dp1_eval:<8} | {dp2_eval:<8} | {dtot_eval:<8}")
    print(f"{'Total DLTs':<18} | {dp1_dlts:<8} | {dp2_dlts:<8} | {dtot_dlts:<8}")
  print("-" * 50)

## Cohort Expansion & Phase 2 Simulations

This function allow you to simulate a trial that first finds the MTD in Phase 1, and then immediately opens a Phase 2 expansion cohort at the MTD (and optionally, dose levels below the MTD).

---

## Viewing a Single Trial (`run_single_expansion`)
Use this to run exactly **one trial** and view a detailed breakdown of patient counts, trial time, and safety metrics for both Phase 1 and the Phase 2 expansion.

### Parameters
* **`trial_type`**: `1` (3+3), `2` (IQ 3+3), `3` (Rolling 6), or `4` (IQ Rolling 6).
* **`expansion_num`**: The total number of patients you want targeted per expansion arm.
* **`replace_ineval`** (default: False):
  * `False`: The trial targets `expansion_num` of patients who will start treatment. If a patient becomes inevaluable, they are not replaced.
  * `True`: The trial targets `expansion_num` of *fully evaluable* patients. If a patient drops out, a new patient is scheduled to replace them.
  * Both options include patients from phase 1 and adhere to their respective rules.
* **`expand_levels_below_mtd`** (default: 0): Expands additional dose cohorts concurrently. `0` means only the MTD is expanded. `1` means the MTD and the dose directly below it are expanded.
* **`config`** (optional): A custom `TrialConfig` object. If left blank, it uses the default trial parameters.
* **`seed`** (optional): Specify an integer seed to reproduce a specific patient pipeline.

### Example Usage
```python
# Runs a 3+3 trial that targets 14 fully evaluable patients at the MTD and 1 level below MTD
run_single_expansion(
    trial_type=1,
    expansion_num=14,
    replace_ineval=True,
    expand_levels_below_mtd=1,
    seed=12345
)

In [25]:
run_single_expansion(trial_type=1, expansion_num=0, replace_ineval=False, expand_levels_below_mtd=1)

--- Single Trial Metrics: 3+3 | Target: 0/DL (Started Treatment) | Multi-DL: 1 levels below | Seed: 151405334781429833469805731784291396487 ---
Result: MTD Declared at Dose Level 3
Expansion Cohorts Activated: Dose Levels [2, 3]

[ Time Taken ]
Phase 1 (To MTD) :  621.2 days (~20.4 months)
Phase 2 Expansion:    0.0 days (~0.0 months)
Total Time       :  621.2 days (~20.4 months)

[ Entire Trial (All Levels) Patient Counts ]
                   | Phase 1  | Phase 2  | Total   
--------------------------------------------------
Started Treatment  | 16       | 0        | 16      
Evaluable          | 14       | 0        | 14      
Inevaluable        | 2        | 0        | 2       
Total DLTs         | 2        | 0        | 2       

[ Dose Level 3 (MTD) Specifics ]
                   | Phase 1  | Phase 2  | Total   
--------------------------------------------------
Started Treatment  | 7        | 0        | 7       
Evaluable          | 6        | 0        | 6       
Total DLTs         |

In [26]:
def get_metrics_expansion(trial_type: int, expansion_num: int, num_repetitions: int, config: TrialConfig, start_seed: int, replace_ineval=False, expand_levels_below_mtd=0):
  metrics = []
  dose_stats = defaultdict(lambda: {"dlts": 0, "total": 0})

  for i in range(num_repetitions):
    # Unpacks phase1_time (but ignores it for batch metrics)
    treated_list, days_taken, phase1_time, final_dose, total_inevals, patient_states = cohort_expansion(
        trial_type, expansion_num, config, seed=start_seed + i, replace_ineval=replace_ineval, expand_levels_below_mtd=expand_levels_below_mtd
    )

    evaluable_list = [p for p in treated_list if p.treatment_outcome != "ineval"]
    num_DLT_above_MTD = 0
    for p in treated_list:
      dose_stats[p.dose]["total"] += 1
      if p.treatment_outcome == "dlt":
        dose_stats[p.dose]["dlts"] += 1
        if final_dose is not None and p.dose > final_dose:
          num_DLT_above_MTD += 1

    metrics.append({
      "days_taken": days_taken,
      "final_dose": final_dose,
      "num_treated": len(treated_list),
      "num_evaluable": len(evaluable_list),
      "num_DLT_above_MTD": num_DLT_above_MTD
    })

  return pd.DataFrame(metrics), dose_stats

In [27]:
def run_batch_report_expansion(scenario_defs, sim_types, expansion_num, num_reps, seed=0, show_evaluable=False, replace_ineval=False, expand_levels_below_mtd=0):
  type_map = {1: "3+3", 2: "IQ 3+3", 3: "Roll 6", 4: "IQ Roll 6"}
  table_rows = []
  base_config = TrialConfig()

  for scenario in scenario_defs:
    valid_fields = {k: v for k, v in scenario.items() if hasattr(base_config, k)}
    config = replace(base_config, **valid_fields)

    row = {"Variation": scenario.get("name", "Unnamed")}

    treated_cols = {}
    eval_cols = {}
    month_cols = {}
    mtd_cols = {}
    safety_summary = []

    for st in sim_types:
      # Use the expansion metrics function
      df, _ = get_metrics_expansion(st, expansion_num, num_reps, config, seed, replace_ineval=replace_ineval, expand_levels_below_mtd=expand_levels_below_mtd)
      df['months'] = df['days_taken'] / 30.4375
      lbl = type_map[st]

      treated_cols[f"#Treated\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['num_treated'])
      month_cols[f"#Months\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['months'])

      if show_evaluable:
        eval_cols[f"#Evaluable\n{lbl}\nMean, Med\n(range)"] = format_stat_str(df['num_evaluable'])

      mtd_cols[f"{lbl}\n% MTD\nAt level"] = format_mtd_distribution(df['final_dose'], config)
      safety_summary.append(f"{lbl}: {df['num_DLT_above_MTD'].mean():.2f}")

    row.update(treated_cols)
    if show_evaluable:
      row.update(eval_cols)
    row.update(month_cols)
    row["Mean\n#DLTs\nabove\nMTD"] = "\n".join(safety_summary)
    row.update(mtd_cols)
    row["DLT rate\n(Ground\nTruth)"] = format_theoretical_dlt_col(config)

    table_rows.append(row)

  print(tabulate(table_rows, headers="keys", tablefmt="grid"))

## Cohort Benchmarks (`run_batch_report_expansion`)
Use this to run simulated trials side-by-side to compare average trial duration, safety profiles, and patient counts.

### Parameters
* **`scenario_defs`**: A list of dictionaries detailing the trial configurations (same format as Phase 1 benchmarking).
* **`sim_types`**: A list of trial types to run (e.g., `[1, 2]` runs standard 3+3 against IQ 3+3).
* **`expansion_num`**: Target cohort size per arm.
* **`num_reps`**: Number of simulated trials to run per configuration (e.g., `800`).
* **`seed`** (default: 0): Starting seed for the batch.
* **`show_evaluable`** (default: False): Toggle to `True` to include the number of evaluable patients in the output table.
* **`replace_ineval`** (default: False): Toggle replacing inevaluable patients to guarantee the target number of fully evaluable patients.
* **`expand_levels_below_mtd`** (default: 0): Number of dose cohorts below the MTD to expand concurrently.

### Example Usage
```python
# 1. Define the scenario variations you want to test
scenarios = [
    {
        "name": "Scenario A\n20% Ineval",
        "ineval_prob": 0.20,
        "starting_dose": 2
    },
    {
        "name": "Scenario B\n3.6% Ineval",
        "ineval_prob": 0.036,
        "starting_dose": 2
    }
]

# 2. Run 800 simulations comparing 3+3 and IQ 3+3
# Targeting 12 fully evaluable patients at MTD only
run_batch_report_expansion(
    scenario_defs=scenarios,
    sim_types=[1, 2],
    expansion_num=12,
    num_reps=800,
    replace_ineval=True,
    expand_levels_below_mtd=0
)
```

In [32]:
scenarios = [
  {
      "name": "Scenario A1\nAve Ineval\nPhase 1\n20% Ineval"
    },
    {
      "name": "Scenario A2\nLow Ineval\nPhase 1\n3.6% Ineval",
      "ineval_prob": 0.036
    },
    {
      "name": "Scenario A3\nHigh Ineval\nPhase 1\n44% Ineval",
      "ineval_prob": 0.44
    },
    {
      "name": "Scenario A4\nAve Ineval\nPhase 1\nScreenfail to 60%",
      "screen_fail_prob": 0.6
    },
    {
      "name": "Scenario A5\nAve Ineval\nPhase 1\nArrival (15 day mean)",
      "mean_interarrival_time": 15.0
    },
    {
      "name": "Scenario A6\nAve Ineval\nPhase 1\nCL to 21 Days",
      "course_length": 21
    },
    {
      "name": "Scenario A7\nAve Ineval\nPhase 1\nToxicity",
      # "dlt_probs": [0.108, 0.136, 0.18, 0.259, 0.403],
      "fifty_rate": 5.5
    }
]

run_batch_report_expansion(
  scenario_defs=scenarios,
  sim_types=[3, 4],
  expansion_num=0,
  num_reps=800,
  replace_ineval=True,
  expand_levels_below_mtd=False
)

+-----------------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------------+-------------+
| Variation             | #Treated    | #Treated    | #Months     | #Months     | Mean            | Roll 6       | IQ Roll 6    | DLT rate    |
|                       | Roll 6      | IQ Roll 6   | Roll 6      | IQ Roll 6   | #DLTs           | % MTD        | % MTD        | (Ground     |
|                       | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above           | At level     | At level     | Truth)      |
|                       | (range)     | (range)     | (range)     | (range)     | MTD             |              |              |             |
+=======================+=============+=============+=============+=============+=================+==============+==============+=============+
| Scenario A1           | 23.6, 25.5  | 23.2, 24.0  | 15.9, 16.5  | 12.7, 12.7  | Roll 6: 0.96    | Fail:   0.6% | Fail:   0.9% | Dose  

#Monotherapy and Combination Therapy Trial


In [29]:
# Monotherapy arm: 2 levels, starts at level 1 (mapped to 2).
mono_scenarios = [
    {
        "name": "Monotherapy\nInterarrival: 10 days",
        "min_dose": 1,
        "starting_dose": 2,
        "max_dose": 2,
        "dlt_probs": [0.05, 0.10],
        "course_length": 21,
        "ineval_prob": 0.10,
        "screen_fail_prob": 0.30,
        "mean_interarrival_time": 10.0
    },
    {
        "name": "Monotherapy\nInterarrival: 14 days",
        "min_dose": 1,
        "starting_dose": 2,
        "max_dose": 2,
        "dlt_probs": [0.05, 0.10],
        "course_length": 21,
        "ineval_prob": 0.10,
        "screen_fail_prob": 0.30,
        "mean_interarrival_time": 14.0
    }
]

print("--- MONOTHERAPY ACCRUAL (PHASE 1 ONLY) ---")
# Using the Phase 1 batch report (assuming no expansion requested for Mono)
run_batch_report(
    scenario_defs=mono_scenarios,
    sim_types=[1, 2], # 3+3 and IQ 3+3
    num_reps=1000
)

--- MONOTHERAPY ACCRUAL (PHASE 1 ONLY) ---
+-----------------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+-------------+
| Variation             | #Treated    | #Treated    | #Months     | #Months     | Mean         | 3+3          | IQ 3+3       | DLT rate    |
|                       | 3+3         | IQ 3+3      | 3+3         | IQ 3+3      | #DLTs        | % MTD        | % MTD        | (Ground     |
|                       | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above        | At level     | At level     | Truth)      |
|                       | (range)     | (range)     | (range)     | (range)     | MTD          |              |              |             |
+=======================+=============+=============+=============+=============+==============+==============+==============+=============+
| Monotherapy           | 6.9, 6.0    | 7.6, 7.0    | 6.4, 5.9    | 5.2, 4.8    | 3+3: 0.25    | Fail:   0.6% |

In [30]:
# Combination Therapy: 5 levels, starts at 1B (mapped to 2).
combo_scenarios = [
    {
        "name": "Combination Therapy\nInterarrival: 10 days",
        "min_dose": 1,
        "starting_dose": 2,
        "max_dose": 5,
        "dlt_probs": [0.05, 0.10, 0.15, 0.20, 0.30],
        "course_length": 42,
        "ineval_prob": 0.20,
        "screen_fail_prob": 0.30,
        "mean_interarrival_time": 10.0
    },
    {
        "name": "Combination Therapy\nInterarrival: 14 days",
        "min_dose": 1,
        "starting_dose": 2,
        "max_dose": 5,
        "dlt_probs": [0.05, 0.10, 0.15, 0.20, 0.30],
        "course_length": 42,
        "ineval_prob": 0.20,
        "screen_fail_prob": 0.30,
        "mean_interarrival_time": 14.0
    }
]

print("\n--- COMBINATION THERAPY (PHASE 1 + PHASE 1 EXPANSION TO 12 TREATED) ---")
# Using the Expansion batch report targeting 12 treated patients
run_batch_report_expansion(
    scenario_defs=combo_scenarios,
    sim_types=[1, 2], # 3+3 and IQ 3+3
    expansion_num=12,
    num_reps=1000,
    replace_ineval=False, # Replaces are OFF because we target 12 "treated", not 12 "evaluable"
    expand_levels_below_mtd=0
)


--- COMBINATION THERAPY (PHASE 1 + PHASE 1 EXPANSION TO 12 TREATED) ---
+-----------------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+-------------+
| Variation             | #Treated    | #Treated    | #Months     | #Months     | Mean         | 3+3          | IQ 3+3       | DLT rate    |
|                       | 3+3         | IQ 3+3      | 3+3         | IQ 3+3      | #DLTs        | % MTD        | % MTD        | (Ground     |
|                       | Mean, Med   | Mean, Med   | Mean, Med   | Mean, Med   | above        | At level     | At level     | Truth)      |
|                       | (range)     | (range)     | (range)     | (range)     | MTD          |              |              |             |
+=======================+=============+=============+=============+=============+==============+==============+==============+=============+
| Combination Therapy   | 22.9, 23.0  | 24.4, 24.0  | 24.9, 24.6  | 20.3, 20.1  |